# PiL-HQUC — Backend Tests + Full IEEE30 Benchmark Suite

Notebook này **chỉ phục vụ kiểm thử và lấy số liệu offline**. Nó thực hiện tuần tự:

1. cài môi trường Qamomile/CUDA-Q trên NVIDIA GPU;
2. chạy toàn bộ test trong `backend/tests` và `benchmark/tests`;
3. chạy đầy đủ cả ba benchmark IEEE30 với `--experiments all`;
4. tạo `benchmark_report.html` và tải bundle kết quả.

Ba benchmark:

- **Benchmark 1:** 8 profile 24 giờ, cùng fleet IEEE30-derived 10 tổ máy, Hybrid `q=10` so với HiGHS;
- **Benchmark 2:** một instance `double-peak`, `q=8,10,14,18,20,24,26`, cùng một HiGHS reference và Top-K=10;
- **Benchmark 3:** `G=10,20,30,40,50`, Hybrid `q=10` và `q=20`, mỗi G có full HiGHS reference.

Mỗi cấu hình quantum có một full warm-up bị loại khỏi thống kê. Notebook này không khởi động FastAPI và không tạo tunnel frontend.


In [ ]:
# 1) Upload and extract the project ZIP
from google.colab import files
from pathlib import Path
import os
import shutil
import zipfile

os.chdir('/content')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f'Upload exactly one PiL-HQUC project ZIP. Received: {zip_names}')

zip_path = Path('/content') / zip_names[0]
extract_dir = Path('/content/pil_hquc_project')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

candidates = sorted({
    backend.parent
    for backend in extract_dir.rglob('backend')
    if backend.is_dir()
    and (backend.parent / 'frontend').is_dir()
    and (backend.parent / 'benchmark').is_dir()
})
if len(candidates) != 1:
    raise RuntimeError(f'Could not identify one project root. Candidates: {candidates}')

ROOT = candidates[0]
BACKEND = ROOT / 'backend'
FRONTEND = ROOT / 'frontend'
BENCHMARK = ROOT / 'benchmark'

print('Project ZIP:', zip_path.name)
print('Project root:', ROOT)
print('Backend:', BACKEND)
print('Frontend:', FRONTEND)
print('Benchmark:', BENCHMARK)


In [ ]:
# 2) Install backend, test and benchmark dependencies
import subprocess
import sys

requirements = BENCHMARK / 'requirements-benchmark.txt'
if not requirements.exists():
    raise FileNotFoundError(requirements)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(requirements)],
    cwd='/content',
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
print('Backend, test and benchmark dependencies installed.')


In [ ]:
# 3) Require CUDA-Q to use the Colab NVIDIA GPU
import os
import subprocess

subprocess.run(['nvidia-smi'], check=True)
os.environ['CUDAQ_TARGET'] = 'nvidia'
os.environ['REQUIRE_CUDAQ'] = '1'

import cudaq
cudaq.set_target('nvidia')
target = cudaq.get_target()
name_value = getattr(target, 'name', str(target))
target_name = name_value() if callable(name_value) else str(name_value)
print('CUDA-Q target:', target_name)
assert 'nvidia' in target_name.lower(), target_name


In [ ]:
# 4) Run the complete backend and benchmark test suites
from pathlib import Path

PYTHON_BIN = '/usr/local/bin/python' if Path('/usr/local/bin/python').exists() else sys.executable
runtime_env = os.environ.copy()
runtime_env['CUDAQ_TARGET'] = 'nvidia'
runtime_env['REQUIRE_CUDAQ'] = '1'
runtime_env['PYTHONPATH'] = os.pathsep.join([str(BACKEND), str(ROOT)])

test_output_path = Path('/content/pil_hquc_test_output.txt')
test_command = [
    PYTHON_BIN, '-m', 'pytest', '-q',
    str(BACKEND / 'tests'),
    str(BENCHMARK / 'tests'),
]
print('Running:', ' '.join(test_command))
with test_output_path.open('w') as test_log:
    completed = subprocess.run(
        test_command,
        cwd=str(ROOT),
        env=runtime_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    test_log.write(completed.stdout)

print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(f'Tests failed. Full output: {test_output_path}')
print('All backend and benchmark tests passed.')


## Chọn chế độ benchmark

`RUN_FULL_BENCHMARK = True` là cấu hình mặc định của notebook này và chạy đầy đủ ba benchmark, toàn bộ seed, 8 scenario, 7 mức qubit và 5 mức generator.

Chỉ đổi thành `False` khi cần kiểm tra nhanh protocol. Chế độ quick vẫn giữ nguyên solver hyperparameters nhưng giảm seed và số data points; không dùng số liệu quick làm kết quả cuối.


In [ ]:
# 5) Run all three IEEE30 benchmarks
RUN_FULL_BENCHMARK = True

benchmark_command = [
    PYTHON_BIN,
    str(BENCHMARK / 'run_all.py'),
    '--experiments', 'all',
]
if not RUN_FULL_BENCHMARK:
    benchmark_command.append('--quick')

print('Mode:', 'FULL' if RUN_FULL_BENCHMARK else 'QUICK PROTOCOL CHECK')
print('Running:', ' '.join(benchmark_command))
subprocess.run(
    benchmark_command,
    cwd=str(ROOT),
    env=runtime_env,
    check=True,
)

report_path = BENCHMARK / 'report' / 'benchmark_report.html'
if not report_path.exists():
    raise FileNotFoundError(report_path)
print('\nAll selected benchmarks completed.')
print('HTML report:', report_path)
print('Raw results:', BENCHMARK / 'results' / 'raw')
print('Summaries:', BENCHMARK / 'results' / 'summary')
print('Figures:', BENCHMARK / 'results' / 'figures')


In [ ]:
# 6) Preview the three summary tables in Colab
import pandas as pd
from IPython.display import display

summary_dir = BENCHMARK / 'results' / 'summary'
summary_files = [
    ('Benchmark 1 — IEEE30 method comparison', summary_dir / 'ieee30_method_comparison_summary.csv'),
    ('Benchmark 2 — Qubit-budget scaling', summary_dir / 'ieee30_qubit_budget_scaling_summary.csv'),
    ('Benchmark 3 — Generator scaling', summary_dir / 'ieee30_generator_scaling_summary.csv'),
]

for title, path in summary_files:
    print('\n' + title)
    if not path.exists():
        print('Missing:', path)
        continue
    display(pd.read_csv(path))


In [ ]:
# 7) Open benchmark_report.html inside Colab
import time
from IPython.display import HTML, display
from google.colab import output

old_server = globals().get('REPORT_SERVER')
if old_server is not None and old_server.poll() is None:
    old_server.terminate()

REPORT_SERVER = subprocess.Popen(
    [PYTHON_BIN, '-m', 'http.server', '8081', '--directory', str(BENCHMARK)],
    cwd='/content',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)
report_base = output.eval_js('google.colab.kernel.proxyPort(8081)')
report_url = f'{report_base}/report/benchmark_report.html'
display(HTML(
    f'<a href="{report_url}" target="_blank" '
    'style="font-size:18px;font-weight:600">Open Full Benchmark Report</a>'
))
print('Report file:', report_path)


In [ ]:
# 8) Download the report, results, figures, metadata, warm-up audit and test output
import shutil
from google.colab import files

results_dir = BENCHMARK / 'results'
report_dir = BENCHMARK / 'report'
if not results_dir.exists() or not report_path.exists():
    raise FileNotFoundError('Run the full benchmark cell before exporting.')

bundle_dir = Path('/content/pil_hquc_full_test_and_benchmark_bundle')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

shutil.copytree(results_dir, bundle_dir / 'results')
shutil.copytree(report_dir, bundle_dir / 'report')
shutil.copy2(test_output_path, bundle_dir / 'backend_and_benchmark_tests.txt')

archive_path = shutil.make_archive(
    '/content/pil_hquc_full_test_and_benchmark_bundle',
    'zip',
    root_dir=bundle_dir,
)

files.download(str(report_path))
files.download(archive_path)
print('Exported:', archive_path)
